# Model 1: Inference with DeepSeek-V3 API

## Goal
Send all 643 questions from the Austrian tax law dataset to the DeepSeek-V3 API and save the answers as a CSV.

## Approach
- DeepSeek-V3 is accessed via API 
- Each question is sent with a system prompt that defines the response style
- Retry logic handles temporary API failures
- Resume logic allows interrupted runs to continue from where they stopped
- Results are saved after every batch

## Requirements
1. Get a DeepSeek API key from: https://platform.deepseek.com/
2. Download dataset_clean.csv from: https://github.com/svakulenk0/wu-llms-ss26/blob/main/dataset_clean.csv
3. Set your API key in the Configuration cell below

In [11]:
!pip install openai pandas --quiet

In [3]:
import os
import math
import time
import pandas as pd
from openai import OpenAI

## 1. Configuration

All parameters are defined here.

In [4]:
# Get your key: https://platform.deepseek.com/
DEEPSEEK_API_KEY = "sk-caa9340ba07849c98db1025241128558" 

CSV_PATH = "./Data/dataset_inference.csv"
OUT_PATH = "../Output/inference_results_deepseek_v3.csv"

MODEL_NAME = "deepseek-chat"  # DeepSeek-V3
BATCH_SIZE = 10    # Number of questions processed before saving progress
MAX_TOKENS = 300  #per answer


# Instructs the model how to behave for every question
SYSTEM_PROMPT = (
    "You are a careful Austrian tax law assistant. "
    "Answer briefly, factually, and clearly in one plain-text paragraph. "
    "Keep your answer under 200 words. "
    "Complete your sentence before stopping. "
    "Do not invent legal references. "
    "Respond in German."
)

print("Input :", CSV_PATH)
print("Output:", OUT_PATH)
print("Model :", MODEL_NAME)

Input : ./Data/dataset_inference.csv
Output: ../Output/inference_results_deepseek_v3.csv
Model : deepseek-chat


## 3. Initialize DeepSeek Client

DeepSeek uses an OpenAI-compatible API, so we can use the standard OpenAI Python client
with a different base URL.

In [5]:
client = OpenAI(
    api_key  = DEEPSEEK_API_KEY,
    base_url = "https://api.deepseek.com"
)

print("Client initialised.")

Client initialised.


## 4. Load Input CSV

The dataset has two columns: `id` and `prompt`.
We clean the text fields to remove whitespace and newlines.

In [6]:
input_df = pd.read_csv(
    CSV_PATH,
    encoding     = "utf-8-sig",
    quotechar    = '"',
    escapechar   = '\\',
    on_bad_lines = 'warn'
)

# Normalise column names and clean text
input_df.columns   = [col.strip().lower() for col in input_df.columns]
input_df           = input_df.dropna(how='all')
input_df['id']     = input_df['id'].astype(str).str.strip()
input_df['prompt'] = input_df['prompt'].astype(str).str.strip()
input_df['prompt'] = input_df['prompt'].str.replace('\n', ' ', regex=False)
input_df['prompt'] = input_df['prompt'].str.replace('\r', ' ', regex=False)

print("Questions loaded:", len(input_df))
print()
print(input_df[['id', 'prompt']].head(3).to_string())

Questions loaded: 643

             id                                                                                                                   prompt
0  CORP-TAX-001                                                  Was ist die steuerliche Bemessungsgrundlage für die Körperschaftsteuer?
1  CORP-TAX-002  Welche steuerlichen Konsequenzen hat es, wenn eine Körperschaft ein zinsloses Darlehen an ihren Gesellschafter vergibt?
2  CORP-TAX-003              Welche Körperschaften sind verpflichtet, sämtliche Einkünfte den Einkünften aus Gewerbebetrieb zuzurechnen?


## 5. Resume Logic

If the output file already exists from a previous run, we skip the questions that were already answered.

In [7]:
if os.path.exists(OUT_PATH):
    existing_df = pd.read_csv(OUT_PATH, encoding="utf-8-sig")
    done_ids    = set(existing_df["id"].astype(str))
    print("Resume:", len(done_ids), "questions already done.")
else:
    existing_df = pd.DataFrame(columns=["id", "answer"])
    done_ids    = set()

work_df = input_df[~input_df["id"].astype(str).isin(done_ids)].copy()
work_df.reset_index(drop=True, inplace=True)

print("Remaining:", len(work_df), "/", len(input_df), "questions")

if len(work_df) == 0:
    print("All questions already processed.")
    raise SystemExit

Remaining: 643 / 643 questions


## 6. API Call Function

This function sends one question to the DeepSeek API and returns the answer.
If the call fails, it retries up to 3 times with an increasing wait between attempts.

In [8]:
def get_answer(prompt, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model       = MODEL_NAME,
                messages    = [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": prompt}
                ],
                max_tokens  = MAX_TOKENS,
                temperature = 0.1,  # Low temperature because factual & consitant
                top_p       = 0.95
            )
            answer = response.choices[0].message.content.strip()
            return " ".join(answer.split())  

        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt  # exponentialbackoff
                print(f"Retry {attempt + 1}/{max_retries} after {wait}s: {str(e)[:60]}")
                time.sleep(wait)
            else:
                print(f"Failed after {max_retries} attempts: {str(e)}")
                return "[ERROR: API call failed]"

## 7. Run Inference

Questions are processed in groups of BATCH_SIZE.
After each group, results are saved to the output file so progress is not lost if the run is interrupted.

Note: the API is called once per question. BATCH_SIZE only controls how often we save, not how many
questions are sent to the API at once.

In [18]:
new_results = []
num_batches = math.ceil(len(work_df) / BATCH_SIZE)
start_time  = time.time()

print("Starting inference on", len(work_df), "questions...")
print()

for batch_idx in range(num_batches):
    batch_start = batch_idx * BATCH_SIZE
    batch_end   = min(batch_start + BATCH_SIZE, len(work_df))
    batch_df    = work_df.iloc[batch_start:batch_end]

    for _, row in batch_df.iterrows():
        answer = get_answer(str(row["prompt"]))
        new_results.append({"id": str(row["id"]), "answer": answer})
        time.sleep(0.1)  # Small delay to stay within API rate limits

    # save progress
    temp_df = pd.concat(
        [existing_df, pd.DataFrame(new_results, columns=["id", "answer"])],
        ignore_index=True
    )
    temp_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

    # Progress logging
    elapsed  = time.time() - start_time
    per_item = elapsed / batch_end
    eta_min  = (per_item * (len(work_df) - batch_end)) / 60
    print(f"Batch {batch_idx + 1}/{num_batches} | {batch_end}/{len(work_df)} done | ETA approx. {eta_min:.1f} min")

print()
print("Inference complete. Results saved to:", OUT_PATH)
print("Total time:", round((time.time() - start_time) / 60, 1), "minutes")

Starting inference on 643 questions...

Batch 1/65 | 10/643 done | ETA approx. 57.3 min
Batch 2/65 | 20/643 done | ETA approx. 56.4 min
Batch 3/65 | 30/643 done | ETA approx. 54.2 min
Batch 4/65 | 40/643 done | ETA approx. 53.4 min
Batch 5/65 | 50/643 done | ETA approx. 59.4 min
Batch 6/65 | 60/643 done | ETA approx. 58.5 min
Batch 7/65 | 70/643 done | ETA approx. 58.0 min
Batch 8/65 | 80/643 done | ETA approx. 56.4 min
Batch 9/65 | 90/643 done | ETA approx. 54.7 min
Batch 10/65 | 100/643 done | ETA approx. 53.6 min
Batch 11/65 | 110/643 done | ETA approx. 52.3 min
Batch 12/65 | 120/643 done | ETA approx. 51.0 min
Batch 13/65 | 130/643 done | ETA approx. 49.7 min
Batch 14/65 | 140/643 done | ETA approx. 48.1 min
Batch 15/65 | 150/643 done | ETA approx. 47.6 min
Batch 16/65 | 160/643 done | ETA approx. 47.1 min
Batch 17/65 | 170/643 done | ETA approx. 46.0 min
Batch 18/65 | 180/643 done | ETA approx. 44.7 min
Batch 19/65 | 190/643 done | ETA approx. 43.5 min
Batch 20/65 | 200/643 done |

## 8. Verify Output

In [19]:
final_df = pd.read_csv(OUT_PATH, encoding="utf-8-sig")

print("Total rows :", len(final_df), "(expected", len(input_df), ")")
print("Columns    :", list(final_df.columns), "(expected ['id', 'answer'])")
print("Complete   :", "YES" if len(final_df) == len(input_df) else "NO - missing rows!")
print()
print("First 5 results:")
print(final_df.head().to_string())

Total rows : 643 (expected 643 )
Columns    : ['id', 'answer'] (expected ['id', 'answer'])
Complete   : YES

First 5 results:
             id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        ans